#### Tab-Net

TabNet is a deep-learning architecture (developed by Google Cloud) designed specifically for tabular datasets.

It uses sequential attention to choose which features to focus on at each decision step. That makes it more interpretable than a plain feed-forward ANN and sometimes better on structured data.

There’s a good open-source implementation called pytorch-tabnet, which makes it easy to try out—kind of like AutoML for tabular data.

In [63]:
import os, math, random
import numpy as np, pandas as pd, torch
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from pytorch_tabnet.tab_model import TabNetRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import QuantileTransformer

from sklearn.dummy import DummyRegressor
import math


import warnings
warnings.filterwarnings("ignore")

In [64]:
import shap, matplotlib
print("NumPy:", np.__version__)       # expect 1.26.x
print("SHAP :", shap.__version__)     # 0.46.0 (or 0.45.1)
print("MPL  :", matplotlib.__version__)


NumPy: 1.26.4
SHAP : 0.46.0
MPL  : 3.10.7


In [65]:
import torch, shap, sklearn, pandas as pd, numpy as np
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__, "| numpy:", np.__version__, "| shap:", shap.__version__)

CUDA available: True
Device: NVIDIA GeForce RTX 4070 Laptop GPU
torch: 2.4.1+cu121 | numpy: 1.26.4 | shap: 0.46.0


In [66]:
# 0) Config (your paths/columns)
DATA_PATH    = r"../../data/entraineddropletfraction_VerticalFlow.csv"
feature_cols = ['D','Usg','UsL','rhol','rhog','st','MuL','Mug']
target_col   = 'e'  # ensure the CSV uses 'e'

In [67]:
#Configs
SEED = 42
rng  = np.random.default_rng(SEED)
torch.manual_seed(SEED)

DATA_PATH = r"../../data/entraineddropletfraction_VerticalFlow.csv"

feature_cols = ['D','Usg','UsL','rhol','rhog','st','MuL','Mug']  # raw 8 inputs per paper
target_col   = 'e'  # ensure your CSV column name matches

#Load data (raw 8 inputs per the paper)
df = pd.read_csv(DATA_PATH)
X = df[feature_cols].to_numpy(dtype=np.float32)
y = df[target_col].to_numpy(dtype=np.float32)

# ==== 3) Helpers (logit + stratification) ====
EPS = 1e-6
def to_logit(yv):
    yv = np.clip(yv, EPS, 1 - EPS)
    return np.log(yv / (1 - yv))

def from_logit(z):
    return 1. / (1. + np.exp(-z))

from sklearn.preprocessing import QuantileTransformer, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.dummy import DummyRegressor

def make_strat_labels(y_cont, n_bins=7, min_per_bin=5):
    for b in range(n_bins, 1, -1):
        qs = np.linspace(0, 1, b+1)
        edges = np.unique(np.quantile(y_cont, qs))
        if len(edges) < 3: 
            continue
        yb = np.digitize(y_cont, edges[1:-1], right=True)
        _, counts = np.unique(yb, return_counts=True)
        if np.all(counts >= min_per_bin):
            return yb
    med = np.median(y_cont)
    return (y_cont > med).astype(int)

def run_tabnet_cv_once(X_mat, y_vec, tabnet_params, scaler_kind="quantile"):
    # choose scaler
    if scaler_kind == "quantile":
        def mk_scaler(n): 
            return QuantileTransformer(output_distribution="normal",
                                       n_quantiles=min(200, n),
                                       random_state=SEED)
    elif scaler_kind == "standard":
        def mk_scaler(n): 
            return StandardScaler()
    else:
        raise ValueError("scaler_kind must be 'quantile' or 'standard'")

    y_bins = make_strat_labels(y_vec, n_bins=7, min_per_bin=N_SPLITS)
    kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    folds = []
    best_rmse = np.inf
    best_model = None

    for fold, (tr, va) in enumerate(kf.split(X_mat, y_bins), 1):
        scaler = mk_scaler(len(tr))
        X_tr = scaler.fit_transform(X_mat[tr]).astype(np.float32)
        X_va = scaler.transform(X_mat[va]).astype(np.float32)
        y_tr, y_va = y_vec[tr].astype(np.float32), y_vec[va].astype(np.float32)

        # baseline
        base = DummyRegressor(strategy="mean").fit(X_tr, y_tr)
        base_rmse = root_mean_squared_error(y_va, base.predict(X_va))
        base_r2   = r2_score(y_va, base.predict(X_va))

        # logit train/eval
        if USE_LOGIT_TARGET:
            y_tr_fit = to_logit(y_tr).reshape(-1,1)
            y_va_fit = to_logit(y_va).reshape(-1,1)
        else:
            y_tr_fit = y_tr.reshape(-1,1)
            y_va_fit = y_va.reshape(-1,1)

        loss_fn = torch.nn.SmoothL1Loss(beta=0.05)

        model = TabNetRegressor(cat_idxs=[], cat_dims=[], **tabnet_params)
        model.fit(
            X_train=X_tr, y_train=y_tr_fit,
            eval_set=[(X_va, y_va_fit)], eval_name=["valid"], eval_metric=["rmse"],
            max_epochs=MAX_EPOCHS, patience=PATIENCE,
            batch_size=BATCH_SIZE, virtual_batch_size=VBS,
            num_workers=0, pin_memory=True, drop_last=False,
            loss_fn=loss_fn,
        )

        pred_fit = model.predict(X_va).ravel().astype(np.float32)
        pred = from_logit(pred_fit) if USE_LOGIT_TARGET else pred_fit

        rmse = root_mean_squared_error(y_va, pred)
        r2   = r2_score(y_va, pred)
        folds.append((r2, rmse))

        if rmse < best_rmse:
            best_rmse = rmse
            best_model = model

    r2s, rmses = zip(*folds)
    return {
        "r2_mean": float(np.mean(r2s)), "r2_std": float(np.std(r2s, ddof=1)),
        "rmse_mean": float(np.mean(rmses)), "rmse_std": float(np.std(rmses, ddof=1)),
    }, best_model

# ---- define a small, safe grid (6 configs) ----
base_params = dict(
    n_d=32, n_a=32, n_steps=4,
    gamma=1.5, n_independent=2, n_shared=2,
    lambda_sparse=3e-4,                 # will also try 1e-4
    optimizer_fn=torch.optim.AdamW,
    optimizer_params=dict(lr=2e-4, weight_decay=1e-4),  # cooler LR
    scheduler_fn=torch.optim.lr_scheduler.ReduceLROnPlateau,
    scheduler_params=dict(mode="min", factor=0.8, patience=25, min_lr=1e-5),
    mask_type="sparsemax",
    verbose=0,
    device_name="cuda" if torch.cuda.is_available() else "cpu",
    seed=SEED,
)

sweep = [
    ("quantile", 3e-4),
    ("quantile", 1e-4),
    ("standard", 3e-4),
    ("standard", 1e-4),
    # two backoff options if things are too hot/cold:
    ("quantile", 3e-4),  # will rerun but with lr 1.5e-4 below
    ("standard", 3e-4),
]

results = []
for i, (scaler_kind, lam) in enumerate(sweep, 1):
    params = base_params.copy()
    params["lambda_sparse"] = lam
    # cool LR further for the last two trials
    if i >= 5:
        params["optimizer_params"] = dict(lr=1.5e-4, weight_decay=1e-4)
    print(f"\n=== Trial {i}: scaler={scaler_kind}, lambda_sparse={lam}, lr={params['optimizer_params']['lr']} ===")
    metrics, _ = run_tabnet_cv_once(X, y, params, scaler_kind=scaler_kind)
    print(f"R²={metrics['r2_mean']:.4f} ± {metrics['r2_std']:.4f} | RMSE={metrics['rmse_mean']:.4f} ± {metrics['rmse_std']:.4f}")
    results.append((scaler_kind, lam, params['optimizer_params']['lr'], metrics))

# Rank by R² (desc)
results_sorted = sorted(results, key=lambda t: t[3]['r2_mean'], reverse=True)
print("\n=== Ranked (best first) ===")
for scaler_kind, lam, lr, m in results_sorted:
    print(f"{scaler_kind:8s} λ={lam:.0e} lr={lr:.1e}  ->  R² {m['r2_mean']:.4f} ± {m['r2_std']:.4f}, RMSE {m['rmse_mean']:.4f} ± {m['rmse_std']:.4f}")




=== Trial 1: scaler=quantile, lambda_sparse=0.0003, lr=0.0002 ===
Stop training because you reached max_epochs = 1500 with best_epoch = 1488 and best_valid_rmse = 0.8944900035858154

Early stopping occurred at epoch 854 with best_epoch = 674 and best_valid_rmse = 0.8987100124359131

Early stopping occurred at epoch 1053 with best_epoch = 873 and best_valid_rmse = 0.9516800045967102

Early stopping occurred at epoch 564 with best_epoch = 384 and best_valid_rmse = 0.9606999754905701

Early stopping occurred at epoch 542 with best_epoch = 362 and best_valid_rmse = 1.0103600025177002
R²=0.6889 ± 0.0235 | RMSE=0.1610 ± 0.0051

=== Trial 2: scaler=quantile, lambda_sparse=0.0001, lr=0.0002 ===

Early stopping occurred at epoch 811 with best_epoch = 631 and best_valid_rmse = 0.9523400068283081

Early stopping occurred at epoch 1040 with best_epoch = 860 and best_valid_rmse = 0.9030900001525879

Early stopping occurred at epoch 639 with best_epoch = 459 and best_valid_rmse = 1.1408400535583496

In [1]:
# ==== 0) Imports ====
import os, math, numpy as np, pandas as pd, torch
from pytorch_tabnet.tab_model import TabNetRegressor

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import QuantileTransformer
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.dummy import DummyRegressor

# ==== 1) Configs ====
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_PATH = r"../../data/entraineddropletfraction_VerticalFlow.csv"
feature_cols = ['D','Usg','UsL','rhol','rhog','st','MuL','Mug']  # raw 8 inputs per paper
target_col   = 'e'  # ensure CSV uses 'e'

# Training knobs (stable)
N_SPLITS   = 5
MAX_EPOCHS = 1500
PATIENCE   = 180
BATCH_SIZE = 256
VBS        = 64
USE_LOGIT_TARGET = True   # bounded target 0–1 benefits from logit training

TABNET_PARAMS = dict(
    n_d=32, n_a=32, n_steps=4,          # capacity that worked best in your sweep
    gamma=1.5, n_independent=2, n_shared=2,
    lambda_sparse=3e-4,                 # best from your sweep
    optimizer_fn=torch.optim.AdamW,
    optimizer_params=dict(lr=2e-4, weight_decay=1e-4),
    scheduler_fn=torch.optim.lr_scheduler.ReduceLROnPlateau,
    scheduler_params=dict(mode="min", factor=0.8, patience=25, min_lr=1e-5),
    mask_type="sparsemax",
    verbose=0,
    device_name=DEVICE,
    seed=SEED,
)

# ==== 2) Load data (raw 8 inputs) ====
df = pd.read_csv(DATA_PATH)
X = df[feature_cols].to_numpy(dtype=np.float32)
y = df[target_col].to_numpy(dtype=np.float32)

# ==== 3) Helpers (logit + stratification) ====
EPS = 1e-6
def to_logit(yv: np.ndarray) -> np.ndarray:
    yv = np.clip(yv, EPS, 1 - EPS)       # ensure strictly in (0,1)
    return np.log(yv / (1 - yv))

def from_logit(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-z))

def make_strat_labels(y_cont, n_bins=7, min_per_bin=N_SPLITS):
    """
    Bin continuous y into quantile bins for StratifiedKFold.
    If any bin is too small, reduce bins until all bins have >= min_per_bin.
    """
    for b in range(n_bins, 2, -1):
        qs = np.linspace(0, 1, b + 1)
        edges = np.unique(np.quantile(y_cont, qs))
        if len(edges) < 3:
            continue
        yb = np.digitize(y_cont, edges[1:-1], right=True)
        _, counts = np.unique(yb, return_counts=True)
        if np.all(counts >= min_per_bin):
            return yb
    # fallback: median split
    med = np.median(y_cont)
    return (y_cont > med).astype(int)

# ==== 4) CV runner (StratifiedKFold, per-fold QuantileTransformer, robust loss) ====
def run_tabnet_cv(X_mat, y_vec, tabnet_params=TABNET_PARAMS):
    y_bins = make_strat_labels(y_vec, n_bins=7, min_per_bin=N_SPLITS)
    kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    fold_metrics = []
    best_model, best_rmse = None, math.inf

    print(f"=== TabNet CV ({DEVICE.upper()}) — Stratified on y ===")
    for fold, (tr, va) in enumerate(kf.split(X_mat, y_bins), 1):
        # Re-seed each fold for stability
        torch.manual_seed(SEED + fold)
        np.random.seed(SEED + fold)

        # Per-fold scaler (Quantile -> Normal) with safe quantiles
        n_q = min(100, len(tr))
        scaler = QuantileTransformer(output_distribution="normal",
                                     n_quantiles=n_q,
                                     random_state=SEED + fold)
        X_tr = scaler.fit_transform(X_mat[tr]).astype(np.float32)
        X_va = scaler.transform(X_mat[va]).astype(np.float32)
        y_tr = y_vec[tr].astype(np.float32)
        y_va = y_vec[va].astype(np.float32)

        # Baseline sanity
        base = DummyRegressor(strategy="mean").fit(X_tr, y_tr)
        base_pred = base.predict(X_va).astype(np.float32)
        base_rmse = root_mean_squared_error(y_va, base_pred)
        base_r2   = r2_score(y_va, base_pred)

        # Train space for target (logit)
        if USE_LOGIT_TARGET:
            y_tr_fit = to_logit(y_tr).reshape(-1, 1)
            y_va_fit = to_logit(y_va).reshape(-1, 1)
        else:
            y_tr_fit = y_tr.reshape(-1, 1)
            y_va_fit = y_va.reshape(-1, 1)

        # Robust loss helps occasional outliers
        loss_fn = torch.nn.SmoothL1Loss(beta=0.05)

        model = TabNetRegressor(cat_idxs=[], cat_dims=[], **tabnet_params)
        model.fit(
            X_train=X_tr, y_train=y_tr_fit,
            eval_set=[(X_va, y_va_fit)], eval_name=["valid"], eval_metric=["rmse"],
            max_epochs=MAX_EPOCHS, patience=PATIENCE,
            batch_size=BATCH_SIZE, virtual_batch_size=VBS,
            num_workers=0, pin_memory=True, drop_last=False,  # Windows/CUDA friendly
            loss_fn=loss_fn,
        )

        # Predict in logit space, then invert to [0,1]
        pred_fit = model.predict(X_va).ravel().astype(np.float32)
        y_pred = from_logit(pred_fit) if USE_LOGIT_TARGET else pred_fit

        rmse = root_mean_squared_error(y_va, y_pred)
        r2   = r2_score(y_va, y_pred)

        print(f"Fold {fold}: BASE R2={base_r2:.4f}, RMSE={base_rmse:.4f} | "
              f"TABNET R2={r2:.4f}, RMSE={rmse:.4f}")

        fold_metrics.append(dict(fold=fold, rmse=rmse, r2=r2,
                                 base_rmse=base_rmse, base_r2=base_r2))
        if rmse < best_rmse:
            best_rmse, best_model = rmse, model

        # Clear CUDA cache between folds for stability on Windows
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    rmses = [m["rmse"] for m in fold_metrics]
    r2s   = [m["r2"] for m in fold_metrics]
    print(f"[TabNet] R² mean={np.mean(r2s):.4f} ± {np.std(r2s, ddof=1):.4f} | "
          f"RMSE mean={np.mean(rmses):.4f} ± {np.std(rmses, ddof=1):.4f}")
    return fold_metrics, best_model

# ==== 5) Run CV ====
fold_metrics, best_model = run_tabnet_cv(X, y)

# ==== 6) Global feature importance ====
fi = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nTop TabNet features:\n", fi)


=== TabNet CV (CUDA) — Stratified on y ===

Early stopping occurred at epoch 789 with best_epoch = 609 and best_valid_rmse = 0.9215400218963623
Fold 1: BASE R2=-0.0000, RMSE=0.2899 | TABNET R2=0.7195, RMSE=0.1535


Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 869 with best_epoch = 689 and best_valid_rmse = 0.909820020198822
Fold 2: BASE R2=-0.0001, RMSE=0.2911 | TABNET R2=0.6947, RMSE=0.1609


Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Stop training because you reached max_epochs = 1500 with best_epoch = 1455 and best_valid_rmse = 1.087090015411377
Fold 3: BASE R2=-0.0000, RMSE=0.2861 | TABNET R2=0.5510, RMSE=0.1917


Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 864 with best_epoch = 684 and best_valid_rmse = 0.8900399804115295
Fold 4: BASE R2=-0.0000, RMSE=0.2862 | TABNET R2=0.7291, RMSE=0.1490


Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 721 with best_epoch = 541 and best_valid_rmse = 0.9195200204849243
Fold 5: BASE R2=-0.0000, RMSE=0.2908 | TABNET R2=0.7207, RMSE=0.1537
[TabNet] R² mean=0.6830 ± 0.0749 | RMSE mean=0.1618 ± 0.0173

Top TabNet features:
 D       0.155562
Usg     0.138779
rhol    0.138758
Mug     0.131774
MuL     0.130416
UsL     0.119367
rhog    0.110435
st      0.074910
dtype: float64


Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
